La méthode développé dans ce fichier pour filtrer la marche est la suivante:
- Segmentation : créer des fenetre de 1s non chevauchantes (non-overlapping 1s window)
- Calcul des 27 caractéristiques temporelles et fréquentielles (temporelles : moyenne, SD, min, max...), retournées dans un tableau
- Traitement des 27 caractéristiques par un Random Forest
- Association du label à la fenetre pour prédire la marche ou non à chaque seconde.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

1. Chargement du signal

In [10]:
folder_path = "C:/Users/roman/Documents/BEaCHILD/Classification/Classification_RCT2/X_et_Y/Data_202_LW.csv" 

raw = pd.read_csv(folder_path, skiprows=7,decimal=",", names = ["Time","Gyro X","Gyro Y","Gyro Z","Accel X","Accel Y","Accel Z","Event","Quat W","Quat X","Quat Y","Quat Z","Unnamed: 12"], dtype = {"Event":str},low_memory=False)
acc_x = raw["Accel X"].values.astype(float)
acc_y = raw["Accel Y"].values.astype(float)
acc_z = raw["Accel Z"].values.astype(float)

# Norme 
signal = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)


2. Calcul des caractéristiques

In [11]:
fs = 128  # fréquence IMU (d'après ton code convert_counts)
window_size_s = 1.0
window_size = int(window_size_s * fs)
step_size = int(window_size * 0.5)

def time_features(x):
    return {
        "min": x.min(),
        "max": x.max(),
        "mean": x.mean(),
        "std": x.std(),
        "cv": x.std()/x.mean() if x.mean()!=0 else 0,
        "p25": np.percentile(x,25),
        "p75": np.percentile(x,75),
        "iqr": np.percentile(x,75) - np.percentile(x,25),
    }

def frequency_features(x):
    feats = {}
    X = rfft(x)
    freqs = rfftfreq(len(x), d=1/fs)
    
    idx = (freqs >= 0.25) & (freqs <= 5.0)
    if idx.sum() == 0:
        return {"dom_freq":0,"dom_mag":0,"entropy":0,"power_ratio":0}
    
    f = freqs[idx]
    m = np.abs(X[idx])
    
    dom = np.argmax(m)
    feats["dom_freq"] = f[dom]
    feats["dom_mag"] = m[dom]
    
    p = m / np.sum(m)
    feats["entropy"] = -np.sum(p * np.log(p + 1e-12))
    
    feats["power_ratio"] = m[dom] / np.sum(m)
    
    return feats

features = []

for start in range(0, len(signal) - window_size, step_size):
    w = signal[start:start+window_size]
    feats = {"start": start / fs}
    feats.update(time_features(w))
    feats.update(frequency_features(w))
    features.append(feats)

features_df = pd.DataFrame(features)
print(features_df.head())


   start       min       max      mean       std        cv       p25  \
0    0.0  0.517400  1.161528  0.990468  0.068303  0.068961  0.955429   
1    0.5  0.517400  1.130392  0.995281  0.062357  0.062653  0.969497   
2    1.0  0.919231  1.111897  0.993579  0.031616  0.031821  0.969233   
3    1.5  0.897680  1.078063  0.992228  0.030015  0.030250  0.967271   
4    2.0  0.428515  1.845616  1.012626  0.173694  0.171529  0.945897   

        p75       iqr  dom_freq   dom_mag   entropy  power_ratio  
0  1.012190  0.056761       5.0  1.717117  1.543519     0.299413  
1  1.016638  0.047142       5.0  1.137118  1.526649     0.276788  
2  1.011598  0.042365       3.0  1.252091  1.529702     0.322333  
3  1.013763  0.046492       3.0  1.641297  1.497278     0.367026  
4  1.045967  0.100070       5.0  5.745852  1.572724     0.263652  


3. Entrainement du Random Forest

4. Association du label à chaque seconde